# [8차시] 원과 부채꼴 — 호의 길이와 넓이

**대연중학교 수학 렌더링 반** | 이름: ____________________  `v1 (2026-06-18)`

📐 **오늘의 수학**: 평면도형 — 원, 부채꼴, 호의 길이, 부채꼴의 넓이

🎯 **학습 목표**

- 원에서 호·현·부채꼴을 구분하고 코드로 그릴 수 있다.
- 중심각이 커질 때 호의 길이와 넓이가 어떻게 변하는지 애니메이션으로 설명할 수 있다.
- 호의 길이·부채꼴 넓이가 중심각에 정비례함을 시각화할 수 있다.


In [ ]:
#@title ⚙️ 1단계: Manim 설치 (재생 ▶ 누르고 약 2분 대기)
# ⚠️ 이 셀은 고치지 말고 그대로 실행하세요. 빨간 경고가 나와도 괜찮아요.
!sudo apt update -qq
!sudo apt install -y -qq libcairo2-dev libpango1.0-dev ffmpeg fonts-nanum > /dev/null
!fc-cache -f > /dev/null
!pip install -q manim
print("📦 설치 끝!")
print("👉 이제 메뉴 [런타임] → [세션 다시 시작] 을 누른 뒤,")
print("   아래 '2단계' 셀부터 실행하세요. (이 셀은 다시 실행하지 않아도 돼요)")


> ### 🔄 잠깐! 설치가 끝나면 꼭 **[런타임] → [세션 다시 시작]** 을 누르세요.
> 새로 설치된 도구를 Colab이 제대로 불러오려면 한 번 재시작해야 해요.
> 재시작 후에는 위의 설치 셀 말고 **아래 셀부터** 실행하면 됩니다.


In [ ]:
#@title ✅ 2단계: 준비 확인 (세션 다시 시작 후 실행)
from manim import *
print("✅ 준비 완료! 이제 아래 셀들을 실행할 수 있어요.")


## 1. 원, 호, 부채꼴 알아보기

원의 일부분을 부르는 이름이 있어요.

| 이름 | 뜻 | Manim 코드 |
|---|---|---|
| 원 | 한 점에서 같은 거리에 있는 점들 | `Circle(radius=2)` |
| 호 (弧) | 원 위 두 점 사이의 **굽은 부분** | `Arc(radius=2, angle=...)` |
| 부채꼴 | 두 반지름과 호로 둘러싸인 **부채 모양** | `AnnularSector(inner_radius=0, outer_radius=2, angle=...)` |
| 중심각 | 부채꼴의 두 반지름이 이루는 각 | 부채꼴의 `angle` |

아래 셀을 실행해서 원 → 호 → 부채꼴 순서로 나타나는 모습을 보세요.


In [ ]:
%%manim -qm -v WARNING CircleArcSector

class CircleArcSector(Scene):
    def construct(self):
        r = 2.5
        ang = 100 * DEGREES        # 중심각 100도

        circle = Circle(radius=r, color=GREY)
        center = Dot(ORIGIN, color=WHITE)

        title = Text("원 → 호 → 부채꼴", font="NanumGothic", font_size=34).to_edge(UP)
        self.play(Write(title))
        self.play(Create(circle), FadeIn(center))
        self.wait(0.5)

        # 호: 굽은 부분만 노란색으로
        arc = Arc(radius=r, start_angle=0, angle=ang, color=YELLOW, stroke_width=8)
        arc_label = Text("호", font="NanumGothic", font_size=28, color=YELLOW).next_to(arc, RIGHT)
        self.play(Create(arc), FadeIn(arc_label))
        self.wait(0.5)

        # 부채꼴: 두 반지름 + 호 안쪽을 색칠
        sector = AnnularSector(inner_radius=0, outer_radius=r, angle=ang,
                               start_angle=0, color=BLUE, fill_opacity=0.5)
        self.play(FadeIn(sector))
        self.wait(1)


## 2. 중심각이 커지면? — 움직이는 부채꼴

부채꼴에서 가장 중요한 사실:

> **호의 길이**와 **부채꼴의 넓이**는 **중심각의 크기에 정비례**해요.

중심각이 2배가 되면 호도 2배, 넓이도 2배! 아래 애니메이션에서 중심각을 0°에서 270°까지 키우면서 숫자가 같이 변하는 걸 확인하세요.

$$\text{호의 길이}=2\pi r\times\frac{\text{중심각}}{360^\circ}, \qquad \text{넓이}=\pi r^2\times\frac{\text{중심각}}{360^\circ}$$


In [ ]:
%%manim -qm -v WARNING SectorGrow

class SectorGrow(Scene):
    def construct(self):
        r = 2.3
        theta = ValueTracker(1)          # 중심각(도) — 0보다 살짝 크게 시작

        circle = Circle(radius=r, color=GREY).shift(LEFT * 2.5)

        # 중심각에 따라 매 순간 다시 그려지는 부채꼴
        sector = always_redraw(lambda: AnnularSector(
            inner_radius=0, outer_radius=r,
            angle=theta.get_value() * DEGREES, start_angle=0,
            color=BLUE, fill_opacity=0.6).shift(LEFT * 2.5))

        # 호만 노란색으로 강조
        arc = always_redraw(lambda: Arc(
            radius=r, start_angle=0, angle=theta.get_value() * DEGREES,
            color=YELLOW, stroke_width=8).shift(LEFT * 2.5))

        # 오른쪽에 실시간 숫자판
        def readout():
            t = theta.get_value()
            arc_len = 2 * PI * r * t / 360
            area = PI * r * r * t / 360
            return Text(f"중심각  {t:.0f}°\n호의 길이  {arc_len:.2f}\n넓이  {area:.2f}",
                        font="NanumGothic", font_size=30, line_spacing=1.2).shift(RIGHT * 3)
        panel = always_redraw(readout)

        self.add(circle, sector, arc, panel)
        self.play(theta.animate.set_value(270), run_time=5, rate_func=linear)
        self.wait(1)


## 3. 부채꼴은 원의 몇 분의 몇? — 피자 나누기

중심각 360°가 원 전체예요. 그래서 부채꼴은 "원의 일부"를 나타내요.

- 중심각 90° → 원의 1/4 (90/360)
- 중심각 120° → 원의 1/3 (120/360)

아래 셀은 원을 여러 조각의 부채꼴(피자!)로 나눠요.


In [ ]:
%%manim -qm -v WARNING PizzaSlices

class PizzaSlices(Scene):
    def construct(self):
        r = 2.6
        n = 6                              # 조각 수 (이 숫자를 바꿔 보세요!)
        colors = [RED, ORANGE, YELLOW, GREEN, BLUE, PURPLE,
                  PINK, TEAL, MAROON, GOLD]

        title = Text(f"원을 {n}조각으로 — 한 조각의 중심각 {360//n}°",
                     font="NanumGothic", font_size=30).to_edge(UP)
        self.play(Write(title))

        slices = VGroup()
        for i in range(n):
            s = AnnularSector(inner_radius=0, outer_radius=r,
                              angle=TAU / n, start_angle=i * TAU / n,
                              color=colors[i % len(colors)], fill_opacity=0.7,
                              stroke_color=WHITE, stroke_width=2)
            slices.add(s)

        self.play(LaggedStart(*[FadeIn(s) for s in slices], lag_ratio=0.3))
        self.wait(0.5)
        # 한 조각만 쏙 빼내기
        self.play(slices[0].animate.shift(RIGHT * 1.2 + UP * 0.7))
        self.wait(1)


## 🏆 도전 과제

아래 셀을 고쳐서 **팩맨(Pac-Man)** 을 만들어 보세요.

1. 입이 벌어진 모습 = 중심각이 360°보다 작은 부채꼴 (예: 300°)
2. 노란색으로 채우고, 눈도 하나 찍어 보기
3. (보너스) `ValueTracker`로 입을 벌렸다 닫았다 움직이기


In [ ]:
%%manim -qm -v WARNING PacMan

class PacMan(Scene):
    def construct(self):
        r = 2.5
        # 1. 입 각도를 바꿔 보세요 (작을수록 입이 크게 벌어져요)
        mouth = 60 * DEGREES

        body = AnnularSector(inner_radius=0, outer_radius=r,
                             angle=TAU - mouth, start_angle=mouth / 2,
                             color=YELLOW, fill_opacity=1)

        # 2. 눈을 찍어 보세요
        eye = Dot([0.2, 1.1, 0], color=BLACK)

        self.play(FadeIn(body))
        self.play(FadeIn(eye))
        # 3. (보너스) 입을 움직이려면 always_redraw 를 써 보세요
        self.wait(1)


## 📝 오늘 배운 것 정리

- **호**는 원의 굽은 부분, **부채꼴**은 두 반지름과 호로 둘러싸인 부분.
- 호의 길이 = `2πr × (중심각/360)`, 부채꼴 넓이 = `πr² × (중심각/360)`.
- 호의 길이·넓이는 **중심각에 정비례** → 중심각 2배면 둘 다 2배.
- `Arc`(호), `AnnularSector`(부채꼴), `ValueTracker`+`always_redraw`(움직이는 도형).

**다음 시간**: 직선만으로 곡선을 그리는 **스트링아트** 영상을 만들어요.
